## 🧠💡 Intelligent Systems  for Smart Health 👨‍⚕👩‍⚕️

# Chest X-Ray Medical Diagnosis with Deep Learning
## Part 1 - Convolutional Neural Networks (CNNs) --> Continued from last session!

Convolutional Neural Networks (**CNN**s) are the most commonly used type of neural network for computer vision tasks. They are particularly well-suited to tasks like image classification and object detection because they can automatically learn and extract relevant features from input images. 

A key feature of convolutional layers show translational invariance. This means that they can detect specific pattern independent of shifts along the data dimensions (say, for a 2D image a pattern is still detected when shifted in x or y-direction).

CNNs consist of multiple layers, each performing a different type of processing on the input data. These layers typically include convolutional layers, which extract features from the input images, and pooling layers, which downsample the output of the convolutional layers. By stacking these layers on top of one another, a CNN can learn increasingly complex representations of the input data.

We will work on the following aspects:
1. Model type and architecture
2. Model goal --> loss function and targets
3. sampling --> what data should the model see?
4. training strategy


## The dataset

For this sesseion, we will be using the [ChestX-ray8 dataset](https://arxiv.org/abs/1705.02315) which contains 108,948 frontal-view X-ray images of about 30,000 unique patients. 
- Each image in the data set contains multiple text-mined labels identifying 14 different pathological conditions. 
- These in turn can be used by physicians to diagnose 8 different diseases. 
- We will use this data to develop a single model that will provide binary classification predictions for each of the 14 labeled pathologies. 
- In other words it will predict 'positive' or 'negative' for each of the pathologies.
 
The full dataset is available for free [here](https://nihcc.app.box.com/v/ChestXray-NIHCC).

**However, we will work with a smaller subset containing about 10% of the original data!!**

In [ ]:
# Import necessary packages
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sb

from tqdm.notebook import tqdm

# fix to matplotlib issue
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

In [ ]:
# Just to check in the beginning --> later we also need the following:
import torch
from torch.utils.data import Dataset
from torchvision.io import read_image, ImageReadMode
from torchvision.transforms import v2

In [ ]:
import warnings

# Suppress FutureWarning messages
warnings.simplefilter(action='ignore', category=FutureWarning)

## 1. Exploration

Read the data from `csv` files.

In [ ]:
path_data = "../../../Data/ChestX_subset/"  # adjust path if necessary

metadata_df = pd.read_csv(os.path.join(path_data, "metadata.csv"))
metadata_df.head()

### 1.1 Data Types and Null Values Check

Run the next cell to explore the data types present in each column and whether any null values exist in the data.
(for instance by using `.info()` or `.decribe()`)

### 1.2 Unique IDs Check

"PatientId" has an identification number for each patient. One thing you'd like to know about a medical dataset like this is if you're looking at repeated data for certain patients or whether each image represents a different person.

In [ ]:
print(
    f"The total patient ids are {metadata_df['patient_id'].count()}, \
from those the unique ids are {len(metadata_df['patient_id'].unique())}."
)

### 1.3 Data Labels

Run the next two code cells to create a list of the names of each patient condition or disease. 

In [ ]:
# Define actual labels (or classes)
non_class_columns = ['image',
                     'follow_up_no',
                     'patient_id',
                     'patient_age',
                     'gender',
                     'view_position']
classes = [c for c in metadata_df.columns if c not in non_class_columns]

# Get the total classes
print(f"There are {len(classes)} classes (or: labels)")
print(f"This includes: {classes}")

In [ ]:
findings = metadata_df[classes].sum()
findings.sort_values(ascending=False)

### 1.4 Data Visualization

Using the image names listed in the csv file, you can retrieve the image associated with each row of data in your dataframe. 

Run the cell below to visualize a random selection of images from the dataset.

In [ ]:
path_images = "../../../Data/ChestX_subset/images_low_resolution/"  # adjust path if necessary

# Pick 3 random images
rng = np.random.default_rng(seed=0)  # reproducible results
random_images = rng.choice(metadata_df.image, 3, replace=False)

# Set up the figure
plt.figure(figsize=(12, 4))  # wider and shorter for a single row

for i, filename in enumerate(random_images):
    img = plt.imread(os.path.join(path_images, filename))
    plt.subplot(1, 3, i + 1)
    plt.imshow(img, cmap='gray')
    plt.axis('off')

# Adjust layout
plt.tight_layout()
plt.show()   

# Split the data!
One of the most fundamental concepts in machine learning is the distinction between *training* and *test* data.
In machine learning, our models "learn" from data. But, to be able to test later how well a model performs we should never use the same data as during training. Models can *overfit*, for instance (partly) learn by heart what to do with certain data points. The model would then only appear to perform well, giving drastically different predictions on new data.

For simpler machine learning scenarios it is often enough to reserve part of the data as a **test set**.

In deep learning, however, we commonly need two different types of test sets: One is called **validation set** and is used to optimize our model and the training, while a second **test set** is kept until the end of an optimization period for a final test of a model.

In [ ]:
metadata_df["atelectasis"].value_counts()

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(metadata_df,
                                     test_size=0.2,
                                     stratify=metadata_df["atelectasis"],
                                     random_state=0
                                    )
print(f"Training set size: {train_df.shape}")
print(f"Test set size: {test_df.shape}")

In [ ]:
# second split

test_df, val_df = train_test_split(test_df,
                                    test_size=0.5,
                                    stratify=test_df["atelectasis"],
                                    random_state=0
                                    )
print(f"Training set size: {train_df.shape}")
print(f"Validation set size: {val_df.shape}")
print(f"Test set size: {test_df.shape}")

In [ ]:
train_df.head()

## 2. Image Preprocessing with Pytorch

Before training, you'll first modify your images to be better suited for training a convolutional neural network. For this task you'll use the Pytorch [transforms](https://pytorch.org/vision/stable/transforms.html) function to perform data preprocessing and data augmentation.

Run the next two cells to import this function and create an image generator for preprocessing.

In [ ]:
from torch.utils.data import DataLoader

# A dataset class for this case is provided in pytorch_utils.py
from pytorch_utils import XRayDataset

---
## Data generator with image transformation
Just to have a clean, **another fresh start** here...

In [ ]:
BATCH_SIZE = 32

classes = metadata_df.columns[6:]

path_images = "../../../Data/ChestX_subset/images_low_resolution/"  # adjust path

In [ ]:
transforms = v2.Compose([
    v2.ToImage(),
    v2.Grayscale(num_output_channels=1),
    v2.Resize(size=(224, 224), antialias=True),
    v2.ToDtype(torch.float32, scale=True)
]
)

In [ ]:
training_data = XRayDataset(
    train_df,
    path_images,
    classes=train_df.columns[6:],
    transform=transforms  # add your custom transformations to each image
)

train_dataloader = DataLoader(training_data, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
x = next(iter(train_dataloader))

In [ ]:
# what is x?

In [ ]:
len(x), x[0].shape, x[1].shape

---
---
# Build a CNN
Here, we design a simple CNN with 5 convolutional layers, each followed by a max-pooling layer.

---
---

In [ ]:
class CNNModel(nn.Module):
    """
    A simple 5‐layer convolutional network followed by two fully connected layers.
    Input: 1×224×224 grayscale image
    Output: num_classes logistic probabilities (sigmoid)
    """
    def __init__(self, num_classes: int):
        super().__init__()

        # Convolutional blocks
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(in_channels=128, out_channels=192, kernel_size=3, padding=1)
        self.conv5 = nn.Conv2d(in_channels=192, out_channels=192, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # After the pool layers, the 224×224 input shrinks:
        # 224→112→56→28→14→7 (so feature map is 192×7x7 after conv5+pool).
        # We flatten that into a vector for the FC layers.
        flattened_size = 192 * 7 * 7

        # Fully connected layers
        self.fc1 = nn.Linear(flattened_size, 64)
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, x):
        # Convolution + ReLU + MaxPool (×4)
        x = F.relu(self.conv1(x))
        x = self.pool(x)

        x = F.relu(self.conv2(x))
        x = self.pool(x)

        x = F.relu(self.conv3(x))
        x = self.pool(x)

        x = F.relu(self.conv4(x))
        x = self.pool(x)

        x = F.relu(self.conv5(x))
        x = self.pool(x)

        # Flatten and feed through FC layers
        x = torch.flatten(x, start_dim=1)
        x = F.relu(self.fc1(x))

        # Use sigmoid because this is a multi‐label problem (each class is independent)
        x = torch.sigmoid(self.fc2(x))
        return x

In [ ]:
# create a new model
model = CNNModel(num_classes=#...)

## Load an already trained model

In [ ]:
path_model = "models/"  # adjust path if necessary
filepath_model = os.path.join(path_model, "cnn_epoch_8.pth")

# if using a GPU the map_location part can be removed
model.load_state_dict(torch.load(filepath_model, weights_only=True, map_location=torch.device('cpu')))

## Evaluate the model --> validation set

In [ ]:
from pytorch_utils import plot_roc_curves

In [ ]:
validation_data = #...

val_dataloader = #...

In [ ]:
predictions_val = []
labels_val = []



In [ ]:
# Convert from torch tensors to numpy arrays
labels_val = np.array(labels_val)
predictions_val = np.array([x.detach().numpy() for x in predictions_val])

---
---
# Transfer Learning

Using large, pretrained CNNs as the basis and make only few necessary adjustments.  
There are many such models that we can download and use: https://docs.pytorch.org/vision/stable/models.html

---
---

In [ ]:
from torchvision import models
from pytorch_utils import train_model


model_ft = #...

## Adapt images to needs of the pretrained model
Typically such models were trained on RGB images and they will then always expect the same input format.

In [ ]:
BATCH_SIZE = 16

transforms = v2.Compose([
    v2.ToImage(),
    v2.Grayscale(num_output_channels=#...),
    v2.Resize(size=(#.....), antialias=True),
    v2.ToDtype(torch.float32, scale=True),
])

training_data = XRayDataset(train_df, path_images, classes=classes, transform=transforms)
train_dataloader = DataLoader(training_data, batch_size=BATCH_SIZE, shuffle=True)

validation_data = XRayDataset(val_df, path_images, classes=classes, transform=transforms)
val_dataloader = DataLoader(validation_data, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
EPOCHS = 10

# Define loss function and optimizer
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model_ft.parameters(), lr=0.001, momentum=0.9)
exp_lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)


## Strategy 2: freeze lower levels
Instead of retraining the full model (this is what the code above does), we can also decide to keep all pretrained parts intact and only train the newly added layers. This is faster because the backpropagation step will not involve the lower layers.

In [ ]:
EPOCHS = 10

# Define loss function and optimizer
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model_ft2.parameters(), lr=0.001, momentum=0.9)
exp_lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

## Load finetuned model

In [ ]:
model_ft = models.resnet18(weights='IMAGENET1K_V1')
num_ftrs = model_ft.fc.in_features
model_ft.fc = nn.Linear(num_ftrs, num_classes)

In [ ]:
path_model = "models/"
filepath_model = os.path.join(path_model, "resnet18_transfer_learning.pth")

# if using a GPU the map_location part can be removed
model_ft.load_state_dict(torch.load(filepath_model, weights_only=True, map_location=torch.device('cpu')))

In [ ]:
predictions_val = []
labels_val = []

# ...

In [ ]:
labels_val = np.array(labels_val)
predictions_val = np.array([x.detach().numpy() for x in predictions_val])